In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.text_cell_render.rendered_html{font size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input{font-family:Consolas; font-size:12pt;}
div.prompt {min width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe {font-size:12px;}
</style>
"""))

# 벡터DB : Chroma vs Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB
- Pinecone : 클라우드 vector DB
    - (https//www.pinecone.io에서 api key 생성 -> .env 추가 (PINECONE_API_KEY 등록)
   

# 0. 패키지 설치

In [2]:
%pip install -q pinecone langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


# 1. knowledge Base 구성을 위한 데이터 생성

In [1]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('data/소득세법(법률)(제21065호)(20260102).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

document_list = loader.load_and_split(text_splitter=text_splitter)
len(document_list)

193

In [2]:
#embedding : OpenAI API text-embeding-3-large
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(model='text-embedding-3-large')

In [3]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os

pc = Pinecone(
    api_key=os.getenv('PINECONE_API_KEY')
)

# 데이터 업로드 할 때
# index_name = 'tax-index'
# database = PineconeVectorStore.from_documents(
#     documents=document_list,
#     embedding=embedding,
#     index_name=index_name
# )

C:\Users\Admin\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CPU times: total: 4.02 s
Wall time: 15.8 s


In [ ]:
#업로드한 벡터 db를 가져올 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)

# 2. 답변 생성을 위한 Retrieval

In [4]:
query = '연봉이 5천만원인 직장인의 소득세는 얼마인가요?'
retrieved_docs = database.similarity_search(query, k=3)

In [6]:
# retrieved_docs[2].page_content
retrieved_doc = '\n\n---\n\n'.join([doc.page_content for doc in retrieved_docs])
print(retrieved_doc)

[전문개정 2009. 12. 31.]



제10조(납세지의 변경신고) 거주자나 비거주자는 제6조부터 제9조까지의 규정에 따른 납세지가 변경된 경우 변경된 날부터 15일 이내에 대통령령으로 정하는 바에 따라 그 변경 후의 납세지 관할 세무서장에게 신고하여야 한다.

[전문개정 2009. 12. 31.]



제11조(과세 관할) 소득세는 제6조부터 제10조까지의 규정에 따른 납세지를 관할하는 세무서장 또는 지방국세청장이 과세한다.

[전문개정 2009. 12. 31.]



제2장 거주자의 종합소득 및 퇴직소득에 대한 납세의무 <개정 2009. 12. 31.>



제1절 비과세 <개정 2009. 12. 31.>



제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31., 2024. 12. 31., 2025. 10. 1., 2025. 12. 23.>

1. 「공익신탁법」에 따른 공익신탁의 이익

2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득

가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득

나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지

In [7]:
retriever = database.as_retriever(
    search_kwargs={'k':3}
)
retrieved_docs = retriever.invoke(query)

In [8]:
# retrieved_docs[2].page_content
retrieved_doc = '\n\n---\n\n'.join([doc.page_content for doc in retrieved_docs])
print(retrieved_doc)

[전문개정 2009. 12. 31.]



제10조(납세지의 변경신고) 거주자나 비거주자는 제6조부터 제9조까지의 규정에 따른 납세지가 변경된 경우 변경된 날부터 15일 이내에 대통령령으로 정하는 바에 따라 그 변경 후의 납세지 관할 세무서장에게 신고하여야 한다.

[전문개정 2009. 12. 31.]



제11조(과세 관할) 소득세는 제6조부터 제10조까지의 규정에 따른 납세지를 관할하는 세무서장 또는 지방국세청장이 과세한다.

[전문개정 2009. 12. 31.]



제2장 거주자의 종합소득 및 퇴직소득에 대한 납세의무 <개정 2009. 12. 31.>



제1절 비과세 <개정 2009. 12. 31.>



제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31., 2024. 12. 31., 2025. 10. 1., 2025. 12. 23.>

1. 「공익신탁법」에 따른 공익신탁의 이익

2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득

가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득

나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지

# 3. 답변 생성

In [9]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano')

In [19]:
# upstatge에서 받은 $20로 llm을 사용하고 싶다면
from langchain_upstage import ChatUpstage
llm = ChatUpstage(
    model='solar-pro2',
    reasoning_effort='high' #느리지만 더 깊게 추론함 (low, medium)
)

In [17]:
prompt=f'''[identity]
- 당신은 최고의 한국 소득세법 전문가입니다
- [context]를 참고해서 사용자의 질문에 답변해 주세요.
- [context]는 다음과 같습니다
{retrieved_doc}
- 질문:{query}'''

In [20]:
ai_message = llm.invoke(prompt)

In [22]:
ai_message.content

'한국 소득세법 및 제공된 [context]를 기반으로 연봉 5천만원인 직장인의 소득세 계산 과정은 다음과 같습니다:\n\n---\n\n### **1. 총급여액 산정**\n- **총급여액**: 5,000만 원 (비과세소득 제외)\n\n---\n\n### **2. 근로소득공제 적용**\n소득세법 제20조 제2항에 따라 근로소득금액은 총급여액에서 **근로소득공제**를 차감한 금액입니다.  \n2023년 기준 근로소득공제율은 다음과 같습니다:\n- 500만 원 이하: 70%  \n- 500만~1,500만 원: 40%  \n- 1,500만~4,500만 원: 15%  \n- 4,500만~1억 원: 5%  \n- 1억 원 초과: 2%  \n\n**계산**:\n- 500만 원 × 70% = **350만 원**  \n- (1,500만 - 500만) × 40% = **400만 원**  \n- (4,500만 - 1,500만) × 15% = **450만 원**  \n- (5,000만 - 4,500만) × 5% = **25만 원**  \n- **총 근로소득공제**: 350 + 400 + 450 + 25 = **1,225만 원**  \n- **근로소득금액**: 5,000만 - 1,225만 = **3,775만 원**\n\n---\n\n### **3. 종합소득 과세표준 산정**\n근로소득금액에서 **기본공제** 등을 차감합니다.  \n- **기본공제**: 150만 원 (모든 납세자 대상)  \n- **과세표준**: 3,775만 - 150만 = **3,625만 원**\n\n---\n\n### **4. 세율 적용 (누진세율)**\n2023년 소득세율표 기준:\n| 과세표준 구간          | 세율 | 누진공제액       |\n|-----------------------|------|------------------|\n| 1,200만 원 이하       | 6%   | -                |\n| 1,200만~4,600만 원    | 15%  | 108만 원        